# **GPT-1 실습**
: Implementation of GPT-1 model, including pre-training & fine-tuning process. Pre-trained on WikiText2, fine-tuned on IMDB Dataset.
## Improving Language Understanding by Generative Pre-Training

⏩ 논문링크: https://www.mikecaptain.com/resources/pdf/GPT-1.pdf


⏩ 깃허브: https://github.com/tony3ynot/GPT-1/blob/main/GPT_1.ipynb 코드를 참고했습니다


📑 추가로 참고해볼만한 깃허브: https://github.com/lyeoni/gpt-pytorch



## 📌 GPT-1 논문 개요: Improving Language Understanding by Generative Pre-Training (Radford et al., 2018)

### 핵심 아이디어
GPT-1은 **두 단계 학습 전략(two-stage training)**을 제안한다.
1. **Pre-training**: 레이블 없는 대용량 텍스트로 언어 모델(Language Model)을 사전학습
2. **Fine-tuning**: 레이블 있는 소규모 데이터로 downstream task에 맞게 미세조정

### BERT와의 차이
| | GPT-1 | BERT |
|---|---|---|
| 구조 | Transformer **Decoder** (단방향) | Transformer **Encoder** (양방향) |
| 사전학습 방식 | **CLM** (Causal Language Modeling) | **MLM** (Masked Language Modeling) |
| 문맥 방향 | 왼쪽 → 오른쪽 | 양방향 |

### 왜 Decoder-only인가?
GPT-1은 "다음 토큰 예측(next-token prediction)" 방식으로 학습하므로,
미래 토큰을 볼 수 없도록 **Causal (Autoregressive) masking**이 필요하다.
→ Encoder의 양방향 attention은 이 조건에 맞지 않음.

In [ ]:
import torch
import torch.nn as nn
from einops import rearrange

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#1. Model Architecture




## 🔧 1-1. Transformer Decoder 구조

### GPT-1이 사용하는 Decoder vs. 원래 Transformer의 Decoder
원래 Transformer(Vaswani et al., 2017)의 Decoder는 **3개의 sub-layer**를 가진다:
1. Masked Multi-Head Self-Attention
2. **Cross-Attention** (Encoder 출력을 참조)
3. Feed-Forward Network

GPT-1은 **seq2seq가 아닌 언어 모델**이므로 Encoder 자체가 없다.
→ Cross-Attention 블록을 제거하고, **Masked Self-Attention + FFN** 2개의 sub-layer만 사용.

### 각 sub-layer의 역할
- **Masked Multi-Head Self-Attention**: 현재 토큰이 자신보다 앞선 토큰들만 참조 (미래 마스킹)
- **Feed-Forward Network (FFN)**: 각 위치(position)에 독립적으로 적용되는 2-layer MLP
- **Layer Normalization + Residual Connection**: 각 sub-layer 뒤에 적용되어 학습 안정화

## 1-1. Transformer Decoder

In [ ]:
### Multi-Head Attention
class MHA(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads

        self.fc_q = nn.Linear(d_model, d_model) # Query
        self.fc_k = nn.Linear(d_model, d_model) # Key
        self.fc_v = nn.Linear(d_model, d_model) # Value

        self.fc = nn.Linear(d_model, d_model) # Linear Layer

        self.scale = torch.sqrt(torch.tensor(d_model/n_heads))

    def forward(self, Q, K, V, mask = None):
        Q = self.fc_q(Q)
        K = self.fc_k(K)
        V = self.fc_v(V)

        ## B = batch size / L = length / H = heads / D = dimension
        # rearrange to implement 'heads'
        Q = rearrange(Q, 'B L (H D) -> B H L D', H = self.n_heads)
        K = rearrange(K, 'B L (H D) -> B H L D', H = self.n_heads)
        V = rearrange(V, 'B L (H D) -> B H L D', H = self.n_heads)

        ## Self-Attention
        # 1. MatMul
        attention_score = Q @ K.transpose(-2, -1)

        # 2. Scale
        attention_score = attention_score / self.scale

        # 3. Masking
        if mask is not None:
            mask = mask.unsqueeze(1).repeat(1, self.n_heads, 1, 1)
            attention_score.masked_fill_(mask, -1e9)

        # 4. SoftMax
        attention_weights = torch.softmax(attention_score, dim=-1)

        # 5. MatMul
        attention = attention_weights @ V

        ## Concat & Linear
        # rearrange to concat
        x = rearrange(attention, 'B H L D -> B L (H D)')
        output = self.fc(x)

        return output


### Feed Forward Network
class FFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.gelu = nn.GELU()

        nn.init.xavier_normal_(self.linear1.weight)
        nn.init.xavier_normal_(self.linear2.weight)

    def forward(self, x):
        x = self.gelu(self.linear1(x))
        output = self.linear2(x)

        return output


### Decoder Layer
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, resid_drop):
        super().__init__()

        self.mha = MHA(d_model, n_heads)
        self.dropout1 = nn.Dropout(resid_drop)
        self.layernorm1 = nn.LayerNorm(d_model, eps=1e-5)

        self.ffn = FFN(d_model, d_ff)
        self.dropout2 = nn.Dropout(resid_drop)
        self.layernorm2 = nn.LayerNorm(d_model, eps=1e-5)

    def forward(self, x, attn_mask):
        # Masked-MHA layer (with residual shortcut connection)
        residual = self.mha(x, x, x, attn_mask)
        residual = self.dropout1(residual)
        x = self.layernorm1(x + residual)

        # FFN layer (with residual shortcut connection)
        residual = self.ffn(x)
        residual = self.dropout2(residual)
        output = self.layernorm2(x + residual)

        return output


### Decoder
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, seq_len, d_model, n_layers, n_heads, d_ff, embd_drop, resid_drop, pad_id):
        super().__init__()

        self.pad_id = pad_id

        ## Decoder Input
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.dropout = nn.Dropout(embd_drop)
        self.pos_embedding = nn.Embedding(seq_len+1, d_model) # learned positional embedding

        ## Decoder Layers
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, resid_drop) for _ in range(n_layers)])

        nn.init.xavier_normal_(self.embedding.weight)

    def forward(self, x):
        ## padding mask for position embedding
        positions = torch.arange(x.size(1), device=x.device).repeat(x.size(0), 1) + 1
        position_pad_mask = x.eq(self.pad_id)
        positions.masked_fill_(position_pad_mask, 0)

        output = self.dropout(self.embedding(x)) + self.pos_embedding(positions)

        ## attention mask
        pad_mask = self.get_padding_mask(x, x, self.pad_id)
        future_mask = self.get_future_mask(x).to(device=pad_mask.device)
        attn_mask = torch.gt((pad_mask.to(dtype=future_mask.dtype) + future_mask), 0)

        for layer in self.layers:
            output = layer(output, attn_mask)

        return output

    ## padding mask : apply masking to padding tokens
    def get_padding_mask(self, q, k, pad_id):
        pad_mask = k.eq(pad_id).unsqueeze(1).repeat(1, q.size(1), 1)

        return pad_mask

    ## future token mask : apply masking to future tokens
    def get_future_mask(self, q):
        bs, q_len = q.size()
        future_mask = torch.ones(bs, q_len, q_len).triu(diagonal=1)

        return future_mask

### 🔍 Masked Multi-Head Attention (MHA)

**Multi-Head Attention의 직관**
- 하나의 attention이 아닌 H개의 head로 분리하여 각기 다른 관점에서 토큰 간 관계를 포착
- 각 head는 d_model/H 차원에서 독립적으로 attention을 계산한 뒤 concat

**Scaled Dot-Product Attention 공식**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- **Q, K, V**: 동일한 입력 x에서 세 개의 선형 변환으로 생성 (Self-Attention이기 때문)
- **√d_k로 나누는 이유**: 차원이 커질수록 내적 값이 커져 softmax gradient가 소실될 수 있기 때문에 스케일 조정

**Causal Masking (Future Masking)**
- GPT-1은 autoregressive 모델이므로 위치 i에서는 위치 i 이후의 토큰을 볼 수 없어야 함
- 구현: attention score에서 미래 위치에 -∞ (≈ -1e9)를 더한 뒤 softmax → 해당 위치의 가중치가 0이 됨
- 코드에서는 upper triangular matrix(`.triu(diagonal=1)`)로 future mask를 생성

**두 가지 마스크**
1. `padding mask`: PAD 토큰에 대한 attention을 막음
2. `future mask`: 미래 토큰에 대한 attention을 막음 (causal)
→ 두 마스크를 OR 연산(`torch.gt(..., 0)`)으로 합산하여 최종 attention mask 구성

### 🔍 Feed-Forward Network (FFN)

**구조**
$$\text{FFN}(x) = W_2 \cdot \text{GELU}(W_1 x + b_1) + b_2$$

- 2개의 선형 변환 사이에 비선형 활성화 함수 삽입
- GPT-1 원 논문 기준: d_model = 768, d_ff = **3072** (= 4 × d_model)
- FFN은 각 위치(position)에 **독립적으로** 적용됨 (position-wise)

**왜 ReLU가 아닌 GELU인가?**
- **ReLU**: 입력 < 0이면 gradient = 0 (dying ReLU 문제)
- **GELU** (Gaussian Error Linear Unit): 입력 값에 확률적 가중치를 적용하는 부드러운 비선형 함수
$$\text{GELU}(x) = x \cdot \Phi(x)$$
  (Φ는 표준 정규분포의 CDF)
- 실제로 NLP 태스크에서 GELU가 ReLU보다 성능이 좋은 것으로 알려짐
- 이후 BERT, GPT-2 등도 GELU 채택

**Xavier 초기화**
- `nn.init.xavier_normal_`를 사용해 weight를 초기화
- 목적: 각 레이어의 입출력 분산을 동일하게 유지하여 학습 초기 gradient 소실/폭발 방지

### 🔍 Positional Embedding

Transformer는 RNN과 달리 순서 정보를 갖지 않으므로 위치 정보를 별도로 주입해야 한다.

**두 가지 방식 비교**
| | 원 Transformer (Vaswani et al.) | GPT-1 |
|---|---|---|
| 방식 | Sinusoidal (고정) | **Learned** (학습 가능) |
| 파라미터 | 없음 | `nn.Embedding(seq_len+1, d_model)` |

**GPT-1의 구현 방식**
- 각 위치 인덱스(1 ~ seq_len)에 대해 학습 가능한 임베딩 벡터를 부여
- PAD 위치는 position = 0으로 마스킹하여 의미 없는 위치 임베딩이 학습에 영향 주지 않도록 함

**입력 구성**
$$\text{input} = \text{Token Embedding}(x) + \text{Position Embedding}(\text{pos})$$
- 두 임베딩을 더한 후 dropout 적용

## 1-2. GPT-1

## 🔧 1-2. GPT-1 모델 헤드 구조

### Pre-training: Language Model Head (GPTLMHead)

**목적함수 L₁ (Causal Language Modeling)**
$$L_1(\mathcal{U}) = \sum_i \log P(u_i \mid u_{i-k}, \ldots, u_{i-1}; \Theta)$$

- 각 타임스텝에서 이전 k개의 토큰을 보고 다음 토큰의 확률을 최대화
- 구현: Transformer Decoder 출력 → Linear Layer → vocab_size 차원의 logit
- **Weight Tying**: `linear.weight = embedding.weight`
  - 출력 선형 레이어의 가중치를 입력 임베딩 가중치와 공유
  - 파라미터 수 절감 + 임베딩 공간의 일관성 유지

---

### Fine-tuning: Classification Head (GPTClsHead)

**GPT-1의 fine-tuning 전략의 핵심: 입력 변환 (Input Transformation)**
- Task-specific한 새 구조 대신, 입력 텍스트를 **특수 토큰으로 감싸는 방식**으로 변환
- 분류 태스크: `[CLS] + text` 형태로 입력
- `[CLS]` 위치의 최종 hidden state를 분류에 사용

**목적함수 L₂ (Supervised Fine-tuning)**
$$L_2(\mathcal{C}) = \sum_{(x,y)} \log P(y \mid x^1, \ldots, x^m)$$

**보조 목적함수 (Auxiliary LM Objective)**
$$L_3(\mathcal{C}) = L_2(\mathcal{C}) + \lambda \cdot L_1(\mathcal{C})$$
- Fine-tuning 중에도 LM loss를 함께 최적화 (λ = 0.5)
- 효과: 일반화 성능 향상 + catastrophic forgetting 완화
- 코드: `loss = cls_loss + (0.5 * lm_loss)`

In [ ]:
### GPT-1
class GPT(nn.Module):
    def __init__(self,
                 vocab_size,
                 seq_len = 512,
                 d_model = 768,
                 n_layers = 12,
                 n_heads = 12,
                 d_ff = 3072,
                 embd_drop = 0.1,
                 resid_drop = 0.1,
                 pad_id = 0):
        super().__init__()

        self.decoder = TransformerDecoder(vocab_size, seq_len, d_model, n_layers, n_heads,
                                          d_ff, embd_drop, resid_drop, pad_id)

    def forward(self, x):
        outputs = self.decoder(x)

        return outputs


### Language Model (pre-training)
class GPTLMHead(nn.Module):
    def __init__(self, gpt):
        super().__init__()
        vocab_size, d_model = gpt.decoder.embedding.weight.size()

        self.gpt = gpt
        self.linear = nn.Linear(d_model, vocab_size, bias = False)
        self.linear.weight = gpt.decoder.embedding.weight

    def forward(self, x):
        x = self.gpt(x)

        lm_logits = self.linear(x)

        return lm_logits


### Classification Model (fine-tuning)
class GPTClsHead(nn.Module):
    def __init__(self, gpt, n_class, cls_token_id, cls_drop=0.1):
        super().__init__()
        vocab_size, d_model = gpt.decoder.embedding.weight.size()
        self.cls_token_id = cls_token_id

        self.gpt = gpt

        # LM
        self.linear1 = nn.Linear(d_model, vocab_size, bias=False)
        self.linear1.weight = gpt.decoder.embedding.weight
        # Cls
        self.linear2 = nn.Linear(d_model, n_class)
        self.dropout = nn.Dropout(cls_drop)

        nn.init.normal_(self.linear2.weight, std=0.02)
        nn.init.normal_(self.linear2.bias, 0)

    def forward(self, x):
        outputs = self.gpt(x)

        lm_logits = self.linear1(outputs)

        outputs = outputs[x.eq(self.cls_token_id)]
        cls_logits = self.linear2(self.dropout(outputs))

        return lm_logits, cls_logits

# 2. Training

## 2-1. Pre-training

## 🔧 2-1. Pre-training

### BPE (Byte-Pair Encoding) Tokenizer

**왜 BPE를 사용하는가?**
- 단어 단위: OOV(Out-of-Vocabulary) 문제 발생
- 문자 단위: 시퀀스가 너무 길어짐
- **BPE**: 자주 등장하는 문자 쌍을 반복적으로 병합하여 subword 단위 어휘 구성

**학습 방식**
1. 모든 단어를 문자 단위로 분리
2. 가장 빈번하게 등장하는 인접 쌍을 하나의 토큰으로 병합
3. 목표 vocab_size에 도달할 때까지 반복

**특수 토큰**
- `<pad>` (id=0): 패딩 토큰 - 짧은 시퀀스를 일정 길이로 맞추기 위해 사용
- `<cls>` (id=1): 분류 토큰 - fine-tuning 시 시퀀스의 시작에 추가, 이 위치의 hidden state로 분류 수행

In [ ]:
!pip install transformers datasets tokenizers

import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import numpy as np
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
### WikiText Dataset class
class WikiTextDataset(Dataset):
    def __init__(self, data, tokenizer, seq_len):
        self.data = data
        self.tokenizer = tokenizer
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]['text']
        encoded = self.tokenizer.encode(text)
        input_ids = encoded.ids

        # sequence length matching
        if len(input_ids) > self.seq_len:
            input_ids = input_ids[:self.seq_len]
        else:
            input_ids = input_ids + [0] * (self.seq_len - len(input_ids))

        # input & target (for next-word prediction)
        inputs = torch.tensor(input_ids[:-1])
        targets = torch.tensor(input_ids[1:])

        return inputs, targets

### GPT-1 원 논문의 하이퍼파라미터

| 항목 | 논문 값 | 이 코드 |
|---|---|---|
| Layers (n_layers) | 12 | 12 |
| d_model | 768 | 768 |
| n_heads | 12 | 12 |
| d_ff | 3072 | 3072 |
| Max seq_len | 512 | 512 |
| Vocab size | 40,478 (BPE) | 10,000 (축소) |
| Batch size | 64 | 8 (자원 제약) |
| Learning rate | 2.5e-4 (Adam) | 5e-5 (AdamW) |
| Epochs | 100 | 3 (축소) |

> ⚠️ 이 실습은 자원 제약으로 인해 vocab_size와 epoch을 줄인 **경량화 버전**이다.
> 원 논문은 BooksCorpus (7,000권 이상의 미출판 도서) 데이터로 학습함.

In [ ]:
### Hyper-parameters
VOCAB_SIZE = 10000
SEQ_LEN = 512
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 5e-5

### Tokenizer Training
dataset = load_dataset('wikitext', 'wikitext-2-raw-v1')
tokenizer = Tokenizer(BPE())
trainer = BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=["<pad>", "<cls>"])
tokenizer.pre_tokenizer = Whitespace()

def get_training_corpus():
    for i in range(0, len(dataset['train'])):
        yield dataset['train'][i]['text']

tokenizer.train_from_iterator(get_training_corpus(), trainer)

### Dataset Setup
train_dataset = WikiTextDataset(dataset['train'], tokenizer, SEQ_LEN + 1)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

### 옵티마이저 & 학습률 스케줄러

**AdamW**
- Adam에서 weight decay를 gradient에 포함하지 않고 **파라미터에 직접 적용**하는 방식
- L2 regularization과 유사하지만 Adam의 adaptive learning rate와 독립적으로 작동
- Transformer 계열 모델 학습에 표준적으로 사용

**Cosine Annealing LR Scheduler**
- 학습률을 코사인 함수 형태로 점진적으로 감소
- 학습 초반에는 큰 학습률로 빠르게 수렴, 후반에는 작은 학습률로 세밀하게 최적화
- GPT-1 논문에서는 learning rate warmup 후 cosine decay 사용

**Gradient Clipping** (`clip_grad_norm_`, max_norm=0.5)
- Gradient norm이 특정 threshold를 초과하면 gradient를 스케일링
- Transformer의 깊은 구조에서 발생할 수 있는 gradient 폭발 방지

In [ ]:
### Model Initialization
model = GPTLMHead(GPT(vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN)).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=len(train_dataloader) * EPOCHS)

### Pre-training의 입출력 구조

**Next-token prediction 방식**

입력 시퀀스: `[t₁, t₂, t₃, ..., tₙ₋₁]`
타겟 시퀀스: `[t₂, t₃, t₄, ..., tₙ]`

- 각 위치 i에서 i+1번째 토큰을 예측
- 코드: `inputs = input_ids[:-1]`, `targets = input_ids[1:]`

**Loss 계산**
- Cross-Entropy Loss (ignore_index=0: PAD 토큰 무시)
- `logits.view(-1, vocab_size)` vs `targets.view(-1)` 형태로 flatten하여 계산
- PAD 위치는 loss에 포함하지 않아야 정확한 학습 가능

In [ ]:
### Pre-Training
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    progress_bar = tqdm(train_dataloader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for batch_idx, (inputs, targets) in enumerate(progress_bar):
        inputs, targets = inputs.to(device), targets.to(device)

        logits = model(inputs)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=0)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': total_loss / (batch_idx + 1)})

    avg_loss = total_loss / len(train_dataloader)
    print(f"\nEpoch {epoch+1} Average Loss: {avg_loss:.4f}")

print("Training completed!")

torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'final_loss': avg_loss
}, '/content/drive/MyDrive/gpt1_pretrained.pt')

Epoch 1/3: 100%|██████████| 4590/4590 [1:03:08<00:00,  1.21it/s, loss=7.13]



Epoch 1 Average Loss: 7.1291


Epoch 2/3: 100%|██████████| 4590/4590 [1:03:15<00:00,  1.21it/s, loss=nan]



Epoch 2 Average Loss: nan


Epoch 3/3:  36%|███▋      | 1671/4590 [23:02<40:31,  1.20it/s, loss=nan]

## 2-2. Fine-tuning

## 🔧 2-2. Fine-tuning

### GPT-1의 Task-specific 입력 변환 전략

GPT-1의 중요한 기여 중 하나는 **다양한 NLP 태스크를 통일된 입력 형식으로 처리**한다는 것이다.

**분류(Classification) 태스크 입력 형식**

[CLS] + text + [PAD] ... [PAD]

- `[CLS]` 토큰의 최종 hidden state → Linear → class 예측
- BERT와 달리 GPT-1에서 `[CLS]`는 **시퀀스의 시작**에 위치
  (BERT는 `[CLS]`를 앞에 두고, GPT-1은 텍스트 끝에 두는 방식도 논문에서 제시되나, 이 코드는 시작에 배치)

**논문에서 제안하는 다른 태스크 입력 형식**
- Textual Entailment: `[CLS] + premise + $ + hypothesis`
- Similarity: 두 방향 모두 입력 후 출력 합산
- Multiple Choice: 각 선택지마다 별도로 인코딩

→ 핵심: 새로운 구조를 추가하지 않고 **특수 토큰 + 입력 변환**만으로 모든 태스크에 적용 가능

In [ ]:
### IMDB Dataset class
class IMDBDataset(Dataset):
    def __init__(self, data, tokenizer, seq_len):
        self.data = data
        self.tokenizer = tokenizer
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]['text']
        label = self.data[idx]['label']

        # Add CLS token at the start
        encoded = self.tokenizer.encode("<cls> " + text)
        input_ids = encoded.ids

        # Truncate or pad sequence
        if len(input_ids) > self.seq_len:
            input_ids = input_ids[:self.seq_len]
        else:
            input_ids = input_ids + [0] * (self.seq_len - len(input_ids))

        return torch.tensor(input_ids), torch.tensor(label)

In [ ]:
### Dataset Setup
from datasets import load_dataset
imdb_dataset = load_dataset('imdb')

train_dataset = IMDBDataset(imdb_dataset['train'], tokenizer, SEQ_LEN)
train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_dataset = IMDBDataset(imdb_dataset['test'], tokenizer, SEQ_LEN)
val_dataloader = DataLoader(val_dataset, batch_size=8)

In [ ]:
premodel = GPTLMHead(GPT(vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN)).to(device)
premodel.load_state_dict(torch.load('gpt1_pretrained.pt')['model_state_dict'])

### Fine-tuning Model Initialization
model = GPTClsHead(
    gpt=premodel.gpt,  # pretrained GPT
    n_class=2,
    cls_token_id=tokenizer.token_to_id("<cls>"),
    cls_drop=0.1
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

### Fine-tuning Loss: Auxiliary LM Objective

**왜 fine-tuning에서도 LM loss를 함께 사용하는가?**

논문에서는 보조 목적함수(auxiliary objective)가 두 가지 효과를 갖는다고 설명한다:
1. **지도 학습 모델의 일반화** 향상
2. **학습 수렴 가속화**

**최종 loss 공식**
$$L_3 = L_2(\text{cls}) + \lambda \cdot L_1(\text{lm})$$

코드 구현:
- `lm_logits`: 입력 시퀀스에 대한 다음 토큰 예측 logit (pre-training과 동일한 방식)
- `cls_logits`: `[CLS]` 위치의 hidden state에서 분류 예측
- `lm_logits[:, :-1]`: 마지막 위치의 logit 제거 → `inputs[:, 1:]`와 길이 맞춤
- λ = 0.5 (논문 권장값)

**Fine-tuning 시 learning rate**
- Pre-training (5e-5)보다 낮은 **1e-5** 사용
- 사전학습된 가중치를 크게 변형하지 않도록 (catastrophic forgetting 방지)

In [ ]:
### Fine-tuning
EPOCHS = 1
auxiliary_ratio = 0.5
best_acc = 0

for epoch in range(EPOCHS):
    ## Training
    model.train()
    total_loss = 0

    progress_bar = tqdm(train_dataloader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for batch_idx, (inputs, labels) in enumerate(progress_bar):
        inputs, labels = inputs.to(device), labels.to(device)

        lm_logits, cls_logits = model(inputs)
        lm_logits = lm_logits[:, :-1].contiguous()

        ## Loss Function w/ Auxiliary Function
        lm_loss = F.cross_entropy(lm_logits.view(-1, lm_logits.size(-1)),
                                  inputs[:, 1:].contiguous().view(-1), ignore_index=0) # L1 (Auxiliary)
        cls_loss = F.cross_entropy(cls_logits, labels) # L2
        loss = cls_loss + (auxiliary_ratio * lm_loss) # L3

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': total_loss / (batch_idx + 1)})

    ## Validation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            _, cls_logits = model(inputs)

            predictions = torch.argmax(cls_logits, dim=-1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total
    print(f"Epoch {epoch+1} Validation Accuracy: {accuracy:.4f}")

    # Save best model
    if accuracy > best_acc:
        best_acc = accuracy
        torch.save(model.state_dict(), '/content/drive/MyDrive/gpt1_imdb_best.pt')
print(f"Fine-tuning completed! Best accuracy: {best_acc:.4f}")

# Test

In [ ]:
model = GPTClsHead(
    GPT(vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN),
    n_class=2,
    cls_token_id=tokenizer.token_to_id("<cls>"),
    cls_drop=0.1
).to(device)

model.load_state_dict(torch.load('gpt1_imdb_best.pt'))
model.eval()

In [ ]:
# Test
def predict_sentiment(text):
    model.eval()
    encoded = tokenizer.encode("<cls> " + text)
    input_ids = encoded.ids

    if len(input_ids) > SEQ_LEN:
        input_ids = input_ids[:SEQ_LEN]
    else:
        input_ids = input_ids + [0] * (SEQ_LEN - len(input_ids))

    inputs = torch.tensor([input_ids]).to(device)

    with torch.no_grad():
        _, cls_logits = model(inputs)
        prediction = torch.argmax(cls_logits, dim=-1)

    return "Positive" if prediction.item() == 1 else "Negative"

# Example
test_text = "This movie was really great! I enjoyed every moment of it."
print(f"Text: {test_text}")
print(f"Sentiment: {predict_sentiment(test_text)}")